# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIR² dataset using the `mlcroissant` library, referencing all data entities by their `@id` fields for maximum reproducibility and clarity.

### Dataset Source
The dataset is described by the [Croissant schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) and contains ordered logistic regression outputs related to knowledge adoption in rangeland management among Kenyan pastoral households.

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant pandas

## 1. Data Loading

Load Croissant metadata and available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load Croissant metadata and assess information
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Examine available record sets, their `@id`s, and the fields and columns within each. All exploration uses Croissant schema `@id`s.

In [ ]:
# List available record sets by @id
record_sets = list(dataset.record_sets)

print("Record set @ids available in this dataset:")
for rs in record_sets:
    print(f"- {rs['@id']}")

# For each record set, list its fields by @id and label
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field']
        if not isinstance(fields, list):
            fields = [fields]
        print("  Fields (@id):")
        for field in fields:
            field_obj = dataset._find_entity(field) if isinstance(field, str) else field
            label = field_obj.get('name', '') if field_obj else ''
            print(f"    - {field_obj['@id']} ({label})")

## 3. Data Extraction

Load each available record set into a Pandas `DataFrame` using the record set `@id`. You can inspect the first few rows and the list of columns (which correspond to field `@id`s).

Below, we extract all record sets whose `@id`s were listed above.

In [ ]:
# Load all record sets found in this dataset
from collections import OrderedDict

dataframes = OrderedDict()

for rs in record_sets:
    record_set_id = rs['@id']
    # Note: set parameters by @id for full reproducibility
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded record set '@id': {record_set_id}")
        print(f"  Columns (field @id): {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

Perform data cleaning and simple exploration by referencing fields using their Croissant `@id` values. Actions include: filtering, normalizing numeric columns, and grouping as an example of preparing the data for analysis.

Please modify the code below to select the desired record set and fields by their `@id`. If there are no numeric fields, update accordingly.

In [ ]:
# Choose a record set for EDA by @id (edit if necessary)
if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    print(f"Using record set @id for EDA: {record_set_id}")
    print(f"Columns (field @id): {df.columns.tolist()}")

    # Identify a numeric field by inspection (update below if necessary)
    sample_numeric = None
    for c in df.columns:
        # Try to infer numeric columns heuristically
        if pd.api.types.is_numeric_dtype(df[c]) or (df[c].dropna().apply(lambda x: str(x).replace('.', '', 1).isdigit()).all()):
            sample_numeric = c
            break

    if sample_numeric is not None:
        numeric_field_id = sample_numeric
        threshold = df[numeric_field_id].astype(float).mean()  # Use mean as sample threshold
        filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
        ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()

        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another available column
        group_field = None
        for c in df.columns:
            if c != numeric_field_id and df[c].nunique() < len(df)//2:
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped by {group_field} (@id):")
            display(grouped_df.head())
        else:
            print("No suitable field found for grouping.")
    else:
        print("No numeric fields detected. Update the notebook when fields are clarified.")
else:
    print("No record sets could be loaded for EDA.")

## 5. Visualization

Visualize numeric field distributions and relationships. Below is a sample histogram and scatter plot, referencing all columns by their Croissant `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check we have suitable data from the previous cell
if len(dataframes) > 0 and sample_numeric is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=20)
    plt.title(f'Distribution of {numeric_field_id} (@id)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Optionally plot two numeric variables if another is present
    numeric_columns = [c for c in df.columns if c != numeric_field_id and pd.api.types.is_numeric_dtype(df[c])]
    if numeric_columns:
        plt.figure(figsize=(6,4))
        sns.scatterplot(x=df[numeric_field_id], y=df[numeric_columns[0]])
        plt.xlabel(numeric_field_id)
        plt.ylabel(numeric_columns[0])
        plt.title(f'Scatterplot: {numeric_field_id} vs. {numeric_columns[0]} (@id)')
        plt.show()

## 6. Conclusion

- This notebook demonstrates a complete data workflow using the FAIR² dataset and the `mlcroissant` standard.
- All record sets and fields are referenced using their `@id` fields for full compliance and traceability.
- You may further adapt the notebook for more advanced statistical analyses or custom visualizations.

**Remember:** All entity selections in this notebook (record sets, fields, etc.) are referenced *by their `@id`* according to best practice for Croissant-aligned datasets.